# Student Notebook 03 — Decision (Layer 3)

Calibrated probability → Greenlight / Pass / Refer using a
cost matrix. The cost asymmetry comes from the project
framing: producing a flop costs $50M, passing on a hit costs
$100M, and a human reader costs $5K.

**Run order:** notebook 02 must run first (we read its
``student_calibrated.joblib``).

**What you'll get out:** per-film recommendations on the cal
set, total cost vs five baselines.

## CONFIG — edit me

The cost matrix is the knob. The defaults come from
``PROJECT_CONTEXT.md`` Section 1. Try halving / doubling
``flop_cost`` to see when the system flips behavior.

In [ ]:
CONFIG = {
    "flop_cost":   50_000_000,    # $50M  — cost of greenlighting a film that flops
    "miss_cost": 100_000_000,    # $100M — cost of passing on a film that becomes a hit
    "refer_cost":      5_000,    # $5K   — cost of one human reader pass
}

## Imports + load

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path("data/processed")
STUDENT = DATA / "student"

bundle = joblib.load(STUDENT / "student_calibrated.joblib")
feat = pd.read_parquet(DATA / "features.parquet").reset_index()
master = pd.read_parquet(DATA / "films_joined.parquet")

df_cal = feat[feat["split"] == "cal"].reset_index(drop=True)
feat_cols = bundle["feature_columns"]
target_col = bundle["config"]["target"]

X_cal = df_cal[feat_cols].fillna(0).values
y_cal = df_cal[target_col].astype(int).values
ids = df_cal["imdb_id"].astype(str).tolist()
proba = bundle["calibrated_model"].predict_proba(X_cal)[:, 1]
print(f"Cal set: {len(df_cal)} films, positive rate {y_cal.mean():.3f}, mean P(hit) {proba.mean():.3f}")

## The decision rule

For each film with calibrated probability ``p``, compute
expected cost of each action and pick the lowest:

- Greenlight expected cost = ``(1 − p) × flop_cost``
- Pass expected cost       = ``p × miss_cost``
- Refer expected cost      = ``refer_cost`` (independent of p)

Tie-break to Refer (the conservative action).

In [ ]:
def decide(p, flop_cost, miss_cost, refer_cost):
    costs = {
        "Greenlight": (1 - p) * flop_cost,
        "Pass":       p * miss_cost,
        "Refer":      refer_cost,
    }
    min_cost = min(costs.values())
    tied = [a for a, c in costs.items() if c == min_cost]
    return "Refer" if "Refer" in tied else tied[0], costs

actions = []
cost_breakdown = []
for p in proba:
    a, c = decide(p, CONFIG["flop_cost"], CONFIG["miss_cost"], CONFIG["refer_cost"])
    actions.append(a)
    cost_breakdown.append(c)
actions = np.array(actions)

# Action distribution.
for a in ["Greenlight", "Pass", "Refer"]:
    print(f"  {a:11s}: {(actions == a).mean()*100:5.1f}% ({(actions == a).sum():>3d} films)")

## System total cost vs five baselines

Each strategy's *realized* cost is computed by looking at the
true label of each cal-set film:

- Greenlight + true positive → $0 (correct).
- Greenlight + true negative → ``flop_cost``.
- Pass + true positive → ``miss_cost``.
- Pass + true negative → $0 (correct).
- Refer (any outcome) → ``refer_cost``.

Lower total cost = better strategy.

In [ ]:
def realized_cost(action, true, flop_cost, miss_cost, refer_cost):
    if action == "Greenlight":
        return 0 if true == 1 else flop_cost
    if action == "Pass":
        return miss_cost if true == 1 else 0
    return refer_cost  # Refer (outcome-independent)

def total_cost(actions, true_labels):
    return sum(realized_cost(a, t, CONFIG["flop_cost"], CONFIG["miss_cost"], CONFIG["refer_cost"])
               for a, t in zip(actions, true_labels))

rng = np.random.default_rng(42)
baselines = {
    "Always-Greenlight": ["Greenlight"] * len(y_cal),
    "Always-Pass":       ["Pass"] * len(y_cal),
    "Read-Everything":   ["Refer"] * len(y_cal),
    "Random":            rng.choice(["Greenlight", "Pass", "Refer"], size=len(y_cal)).tolist(),
    "System (you!)":     actions.tolist(),
}
rows = []
for name, acts in baselines.items():
    t = total_cost(acts, y_cal)
    rows.append({"strategy": name, "total_cost_USD": t,
                 "cost_per_film_M": t / len(y_cal) / 1e6})
bench = pd.DataFrame(rows).sort_values("total_cost_USD")
print(bench.to_string(index=False))

## Bar chart (log scale because the scales differ wildly)

In [ ]:
bench_sorted = bench.sort_values("total_cost_USD", ascending=True)
colors = ["#2c7fb8" if s == "System (you!)" else "#a6cee3" for s in bench_sorted["strategy"]]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(bench_sorted["strategy"], bench_sorted["total_cost_USD"].clip(lower=1), color=colors)
ax.set_xscale("log")
ax.set_xlabel("Total realized cost on cal set (USD, log scale)")
ax.set_title("Cost-asymmetric decision: system vs baselines")
for s, v in zip(bench_sorted["strategy"], bench_sorted["total_cost_USD"]):
    ax.text(max(v, 1) * 1.2, s, f"${v/1e6:.1f}M" if v >= 1e6 else f"${v/1e3:.0f}K", va="center", fontsize=9)
plt.show()

## Save per-film decisions

In [ ]:
master_lookup = master.set_index("imdb_id")["movie_name"]
decisions_df = pd.DataFrame({
    "imdb_id":                ids,
    "movie_name":             [master_lookup.get(i, i) for i in ids],
    "calibrated_probability": proba,
    "true_label":             y_cal,
    "recommended_action":     actions,
    "expected_cost_GL":       [c["Greenlight"] for c in cost_breakdown],
    "expected_cost_Pass":     [c["Pass"]       for c in cost_breakdown],
    "expected_cost_Refer":    [c["Refer"]      for c in cost_breakdown],
})
decisions_df.to_csv(STUDENT / "student_decisions.csv", index=False)
print(f"Saved {STUDENT / 'student_decisions.csv'}: {len(decisions_df)} rows")
# Show the few films the system recommends Greenlight on.
print("\nGreenlight films (if any):")
gl = decisions_df[decisions_df["recommended_action"] == "Greenlight"]
print(gl[["movie_name", "calibrated_probability", "true_label"]].to_string(index=False) if len(gl)
      else "  (none — under the default cost matrix the model rarely commits)")

## Compare to your teammates

Try halving the refer cost (``"refer_cost": 2_500``) — the
system should commit to more Greenlights. Try raising it to
$1M — the system will refuse to commit on anything. The
knob to play with most is ``refer_cost``: it represents
your studio's effective per-film human-reader cost.

Reference: under the source default cost matrix the rigorous
Phase 6 pipeline commits 1.2% Greenlight and 98.8% Refer on
the cal set, with total cost $1.3M (tying Read-Everything).
You should land in that neighborhood.